## Significance testing of the seasonal UHI by obs/simobs

Goal is to answer the following question: 
 
 Is the UHI extracted from the daily temperature values of a given observation/simulated observation, during a specific season, statistically significant?

#### Steps:
1. Define UHI by season, year, daily temperature value:
    
    - To include non-continuous temperatures (like minimum or maximum daily), or season filtering, consider a binary data mask $\delta$.
    
    $$\delta(t) = \begin{cases}
        1 & \text{ when } T(\vec{x},t) \text{ exists} \\
        0 & \text{ otherwise }
    \end{cases}$$

    - Averaging a temperature over all urban and rural spatial regions (or specific station points) gives $^u\overline{T}(t)$ and $^r\overline{T}(t)$ respectively.
    - A general formula for the time average UHI in over some time interval $\Delta t = t_f - t_0$ emerges.
    
    $$ ^{\Delta t}\overline{U}(T) = \frac{1}{\int^{t_f}_{t_0} \delta(t) dt} \int^{t_f}_{t_0} [^u\overline{T}(t) - ^r\overline{T}(t)]\delta(t) dt $$

    - More practically, with $N$ number of days in which a daily $T$ is sampled, only including days with over 80% station availability for both urban and rural categories, then given the the $i^{\text{th}}$ valid sample day of $\Delta t$ meeting these criteria: 
    $$ ^{\Delta t}\overline{U}(T) = \frac{1}{N} \sum ^{N}_{i=1}[^u\overline{T}(t_i) - ^r\overline{T}(t_i)]$$

2.  Define sampling: 
    - To address the non-independence of $N$ daily samples, for the $Y^\text{th}$ assumed-independent year $\Delta t_Y$ is taken at sub-annual time scales.Uncertainty is therefore estimated as the inter-annual variability of seasonal means over $n=23$ total years of data. Given a variance $\sigma^2$ corresponding to $n = 23$ values of $^{\Delta t_Y}\overline{U}(T)$, a standard error $\text{SE}$ is defined.

    $$\text{SE} = \frac{\sigma}{\sqrt{n}}$$

    - This method should filter out synoptic signals which would otherwise impose an artificial inflation of the number of independent samples and assumes inter-annual signals within the data are not colinear.
    
3. Define null and alternative hypotheses and confidence for each obs/simobs set, season, and type of daily temperature used:
    - $H_0:$  $ ^{\Delta t}\overline{U}(T) = 0\text{°C}$
    - $H_a:$  $ ^{\Delta t}\overline{U}(T) \ne 0\text{°C}$
    - $\alpha = 0.05 $

4. From the $p$-value, given a 2-sided $t$-test at a $1-\alpha = 95\%$ confidence, the error range for about each season/model mean $^{\Delta t}\overline{U}(T)$ over the full time domain emerges. For modelled error ranges which overlap with observation error ranges the bias is set to be "within range", otherwise the UHI is considered to be "cold" or "warm" with respect to the UHI calculated by the station data.

In [1]:
from UHI_statistics import UHI_daily, UHI_seasonal
import numpy as np

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [2]:
UHI = UHI_seasonal # in the form UHI[field = tasmin/tasmax][model = S/C/T (for stations, CLASS or TEB+CLASS)][season = JJA/SON/DJF/MAM]
# Loop to more cleanly print/display UHI statistics
for f in ['tasmin','tasmax']:
    for s in ['JJA','SON','DJF','MAM']:
        for m in ['S','C','T']:
            bias = 'within range'
            
            if m == 'S':
                print(f'{f}_{m}_{s}: [{np.round(UHI[f][m][s]['UHI'],2)} ± {np.round(UHI[f][m][s]['ERR'],4)}]°C | p<0.05:{UHI[f][m][s]['PVAL']<0.05}')
            else:
                obs_range = [ UHI[f]['S'][s]['UHI'] - UHI[f]['S'][s]['ERR'],    UHI[f]['S'][s]['UHI'] + UHI[f]['S'][s]['ERR'] ]
                sim_range = [ UHI[f][m][s]['UHI'] - UHI[f][m][s]['ERR']    ,    UHI[f][m][s]['UHI'] + UHI[f][m][s]['ERR']     ]
                lower = min(obs_range[0],sim_range[0])
                within_range = False 
                for i in [0,1]:
                    within_range = within_range or (sim_range[i] >= obs_range[0] and sim_range[i] <= obs_range[1]) or (obs_range[i] >= sim_range[0] and obs_range[i] <= sim_range[1])
                # within_range = (obs_range[0] <= sim_range[0] and obs_range[1] >= obs_range) or
                if not within_range:
                    if (UHI[f][m][s]['UHI'] - UHI[f]['S'][s]['UHI'] < 0):
                        bias = 'underestimate'
                    else:
                        bias = 'overestimate'
                UHI[f][m][s]['bias'] = bias

                print(f'{f}_{m}_{s}: [{np.round(UHI[f][m][s]['UHI'],2)} ± {np.round(UHI[f][m][s]['ERR'],4)}]°C | p<0.05:{UHI[f][m][s]['PVAL']<0.05} | bias: {bias}')
        print()

tasmin_S_JJA: [2.15 ± 0.1276]°C | p<0.05:True
tasmin_C_JJA: [0.21 ± 0.0628]°C | p<0.05:True | bias: underestimate
tasmin_T_JJA: [2.05 ± 0.0873]°C | p<0.05:True | bias: within range

tasmin_S_SON: [1.91 ± 0.1431]°C | p<0.05:True
tasmin_C_SON: [-0.16 ± 0.0539]°C | p<0.05:True | bias: underestimate
tasmin_T_SON: [1.13 ± 0.0475]°C | p<0.05:True | bias: underestimate

tasmin_S_DJF: [1.82 ± 0.2164]°C | p<0.05:True
tasmin_C_DJF: [-0.17 ± 0.0673]°C | p<0.05:True | bias: underestimate
tasmin_T_DJF: [1.05 ± 0.0684]°C | p<0.05:True | bias: underestimate

tasmin_S_MAM: [1.5 ± 0.1898]°C | p<0.05:True
tasmin_C_MAM: [-0.34 ± 0.0581]°C | p<0.05:True | bias: underestimate
tasmin_T_MAM: [1.23 ± 0.0736]°C | p<0.05:True | bias: underestimate

tasmax_S_JJA: [0.12 ± 0.0959]°C | p<0.05:True
tasmax_C_JJA: [-0.68 ± 0.1124]°C | p<0.05:True | bias: underestimate
tasmax_T_JJA: [0.68 ± 0.0293]°C | p<0.05:True | bias: overestimate

tasmax_S_SON: [-0.11 ± 0.0694]°C | p<0.05:True
tasmax_C_SON: [-0.76 ± 0.0885]°C | p<

Summary:
___
 - A 2-sided t-test at a 95% confidence was used to predict the existence of the urban heat island effect on daily minimum and maximum temperatures, extracted by averaging records of urban and rural station locations within the region closest to Montréal, for each season over 23 years of observation and CRCM6 model data (including or excluding TEB in the surface scheme). 
 - The hypothesis that the UHI is 0°C was tested for each season (JJA,MAM,SON,DJF) model/observation (**S**tation data,**C**LASS,**T**EB+CLASS) and daily temperature record (tasmin,tasmax).
 - Overlapping ranges with model data was used to evaluate a cold or warm bias of any modelled UHI.

 Comments:
 ___
Station data:
 - The null hypothesis is rejected for all seasons in station data except for tasmax in MAM, meaning that we are unable to reject the non-existence of the springtime UHI effect on maximum daily temperature from station observations.
 - Station data suggests a slightly negative, yet still nonzero within confidence, UHI for SON maximum daily temperature.

Simulated observations:
 - Modelled data tends to be more consistent with station data for minimum temperatures in summer months. 
    - TEB+CLASS gives type 2 added value compared to CLASS alone, this is despite a cold bias in TEB for SON & DJF.
        - This bias is still less pronounced than the year-round cold bias from CLASS alone. 
    - CLASS rejects the null hypothesis suggesting a nonzero/negative tasmin UHI for MAM 
        - TEB+CLASS and station data are both consistent in reflecting a nonzero/positive UHI for the same period.

Overall, the inclusion of TEB tends to reduce the annual cold bias of minimum daily temperatures and better simulate its UHI with respect to station records with which it is consistent during spring (MAM) and summer (JJA). CLASS alone predicts a negative heat island as often as not for minimum daily temperatures, where observation and TEB+CLASS show the strongest positive UHI.

With the exception of CLASS, the UHI for daily temperatures seems to be best captured for the minimum daily temperatures. 